## CME APR 2025

In [1]:
import pandas as pd
import numpy as np

cme_apr_2025 = pd.read_csv("../../data/raw/CME/CME_APR_2025.csv")
cme_apr_2025 = cme_apr_2025.reset_index(drop = True)
cme_apr_2025["time"] = pd.to_datetime(cme_apr_2025["time"],unit="s") 

In [2]:
cme_apr_2025

,time,open,high,low,close,Volume
0,2024-11-24 23:00:00,95.740,95.760,95.740,95.755,141
1,2024-11-24 23:01:00,95.755,95.755,95.755,95.755,16
2,2024-11-24 23:03:00,95.755,95.755,95.755,95.755,9
3,2024-11-24 23:04:00,95.755,95.755,95.755,95.755,4
4,2024-11-24 23:09:00,95.755,95.755,95.755,95.755,4
...,...,...,...,...,...,...
20062,2025-04-30 17:37:00,95.670,95.670,95.670,95.670,27
20063,2025-04-30 17:56:00,95.670,95.670,95.670,95.670,33
20064,2025-04-30 18:23:00,95.670,95.670,95.670,95.670,30
20065,2025-04-30 20:46:00,95.670,95.670,95.670,95.670,95


## CME MAY 2025

In [3]:
cme_may_2025 = pd.read_csv("../../data/raw/CME/CME_MAY_2025.csv")
cme_may_2025 = cme_may_2025.reset_index(drop = True)
cme_may_2025["time"] = pd.to_datetime(cme_may_2025["time"],unit="s") 

In [4]:
cme_may_2025

,time,open,high,low,close,Volume
0,2025-01-01 23:00:00,95.8600,95.8600,95.8600,95.8600,6
1,2025-01-01 23:20:00,95.8550,95.8550,95.8550,95.8550,17
2,2025-01-01 23:32:00,95.8550,95.8550,95.8550,95.8550,65
3,2025-01-01 23:58:00,95.8550,95.8550,95.8550,95.8550,2
4,2025-01-02 00:00:00,95.8550,95.8550,95.8550,95.8550,21
...,...,...,...,...,...,...
23602,2025-05-30 14:55:00,95.6700,95.6700,95.6700,95.6700,1
23603,2025-05-30 15:15:00,95.6725,95.6725,95.6725,95.6725,2
23604,2025-05-30 15:16:00,95.6725,95.6725,95.6725,95.6725,8
23605,2025-05-30 15:32:00,95.6700,95.6700,95.6700,95.6700,1


## Merge and then calculate probabilities

In [5]:
apr = cme_apr_2025.copy()
apr["time"] = pd.to_datetime(apr["time"], unit="s")
apr = apr.sort_values("time").set_index("time")

apr = apr.add_suffix("_apr_2025")

may = cme_may_2025.copy()
may["time"] = pd.to_datetime(may["time"], unit="s")
may = may.sort_values("time").set_index("time")

may = may.add_suffix("_may_2025")

apr_aligned = apr.reindex(may.index, method="ffill")


final_df = pd.concat([may, apr_aligned], axis=1)
final_df = final_df.reset_index()

In [6]:
final_df

,time,open_may_2025,high_may_2025,low_may_2025,close_may_2025,Volume_may_2025,open_apr_2025,high_apr_2025,low_apr_2025,close_apr_2025,Volume_apr_2025
0,2025-01-01 23:00:00,95.8600,95.8600,95.8600,95.8600,6,95.815,95.815,95.815,95.815,6
1,2025-01-01 23:20:00,95.8550,95.8550,95.8550,95.8550,17,95.815,95.815,95.815,95.815,6
2,2025-01-01 23:32:00,95.8550,95.8550,95.8550,95.8550,65,95.815,95.815,95.815,95.815,6
3,2025-01-01 23:58:00,95.8550,95.8550,95.8550,95.8550,2,95.815,95.815,95.815,95.815,6
4,2025-01-02 00:00:00,95.8550,95.8550,95.8550,95.8550,21,95.810,95.810,95.810,95.810,58
...,...,...,...,...,...,...,...,...,...,...,...
23602,2025-05-30 14:55:00,95.6700,95.6700,95.6700,95.6700,1,95.670,95.670,95.670,95.670,82
23603,2025-05-30 15:15:00,95.6725,95.6725,95.6725,95.6725,2,95.670,95.670,95.670,95.670,82
23604,2025-05-30 15:16:00,95.6725,95.6725,95.6725,95.6725,8,95.670,95.670,95.670,95.670,82
23605,2025-05-30 15:32:00,95.6700,95.6700,95.6700,95.6700,1,95.670,95.670,95.670,95.670,82


## Calculating Implied Probabilities

In [7]:
meeting_day = 7
days_in_may = 31

N = meeting_day           
M = days_in_may - N        

imp_df = pd.DataFrame()
imp_df["time"] = final_df["time"].copy()

imp_df["effr_avg_apr"] = 100 - final_df["close_apr_2025"]
imp_df["effr_avg_may"] = 100 - final_df["close_may_2025"]

imp_df["effr_end_apr"] = imp_df["effr_avg_apr"]
imp_df["effr_start_may"] = imp_df["effr_end_apr"]
imp_df["volume"] = final_df["Volume_may_2025"]


In [8]:
imp_df

,time,effr_avg_apr,effr_avg_may,effr_end_apr,effr_start_may,volume
0,2025-01-01 23:00:00,4.185,4.1400,4.185,4.185,6
1,2025-01-01 23:20:00,4.185,4.1450,4.185,4.185,17
2,2025-01-01 23:32:00,4.185,4.1450,4.185,4.185,65
3,2025-01-01 23:58:00,4.185,4.1450,4.185,4.185,2
4,2025-01-02 00:00:00,4.190,4.1450,4.190,4.190,21
...,...,...,...,...,...,...
23602,2025-05-30 14:55:00,4.330,4.3300,4.330,4.330,1
23603,2025-05-30 15:15:00,4.330,4.3275,4.330,4.330,2
23604,2025-05-30 15:16:00,4.330,4.3275,4.330,4.330,8
23605,2025-05-30 15:32:00,4.330,4.3300,4.330,4.330,1


In [9]:
imp_df["effr_end_may"] = (
    imp_df["effr_avg_may"] - (N / (N + M)) * imp_df["effr_start_may"]
) / (M / (N + M))

In [10]:
imp_df["delta_effr"] = imp_df["effr_end_may"] - imp_df["effr_start_may"]
imp_df["num_hikes"] = imp_df["delta_effr"] / 0.25

imp_df["hikes_floor"] = np.floor(imp_df["num_hikes"])
imp_df["hikes_decimal"] = imp_df["num_hikes"] - imp_df["hikes_floor"]

states = [-4, -3, -2, -1, 0, 1, 2, 3, 4]

state_to_col = {
    -4: "prob_cut_100",
    -3: "prob_cut_75",
    -2: "prob_cut_50",
    -1: "prob_cut_25",
     0: "prob_no_change",
     1: "prob_hike_25",
     2: "prob_hike_50",
     3: "prob_hike_75",
     4: "prob_hike_100"
}

for col in state_to_col.values():
    imp_df[col] = 0.0

for i, row in imp_df.iterrows():
    floor = int(row["hikes_floor"])
    decimal = row["hikes_decimal"]
    
    p_low = 1 - decimal
    p_high = decimal
    
    if floor in state_to_col:
        imp_df.at[i, state_to_col[floor]] += p_low

    if (floor + 1) in state_to_col:
        imp_df.at[i, state_to_col[floor + 1]] += p_high

In [11]:
imp_df

,time,effr_avg_apr,effr_avg_may,effr_end_apr,effr_start_may,volume,effr_end_may,delta_effr,num_hikes,hikes_floor,hikes_decimal,prob_cut_100,prob_cut_75,prob_cut_50,prob_cut_25,prob_no_change,prob_hike_25,prob_hike_50,prob_hike_75,prob_hike_100
0,2025-01-01 23:00:00,4.185,4.1400,4.185,4.185,6,4.126875,-0.058125,-0.232500,-1.0,0.767500,0.0,0.0,0.0,0.232500,0.767500,0.0,0.0,0.0,0.0
1,2025-01-01 23:20:00,4.185,4.1450,4.185,4.185,17,4.133333,-0.051667,-0.206667,-1.0,0.793333,0.0,0.0,0.0,0.206667,0.793333,0.0,0.0,0.0,0.0
2,2025-01-01 23:32:00,4.185,4.1450,4.185,4.185,65,4.133333,-0.051667,-0.206667,-1.0,0.793333,0.0,0.0,0.0,0.206667,0.793333,0.0,0.0,0.0,0.0
3,2025-01-01 23:58:00,4.185,4.1450,4.185,4.185,2,4.133333,-0.051667,-0.206667,-1.0,0.793333,0.0,0.0,0.0,0.206667,0.793333,0.0,0.0,0.0,0.0
4,2025-01-02 00:00:00,4.190,4.1450,4.190,4.190,21,4.131875,-0.058125,-0.232500,-1.0,0.767500,0.0,0.0,0.0,0.232500,0.767500,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23602,2025-05-30 14:55:00,4.330,4.3300,4.330,4.330,1,4.330000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,1.000000,0.0,0.0,0.0,0.0
23603,2025-05-30 15:15:00,4.330,4.3275,4.330,4.330,2,4.326771,-0.003229,-0.012917,-1.0,0.987083,0.0,0.0,0.0,0.012917,0.987083,0.0,0.0,0.0,0.0
23604,2025-05-30 15:16:00,4.330,4.3275,4.330,4.330,8,4.326771,-0.003229,-0.012917,-1.0,0.987083,0.0,0.0,0.0,0.012917,0.987083,0.0,0.0,0.0,0.0
23605,2025-05-30 15:32:00,4.330,4.3300,4.330,4.330,1,4.330000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,1.000000,0.0,0.0,0.0,0.0


In [12]:
imp_df["time"] = pd.to_datetime(imp_df["time"], utc=True)

## Filtering Based on Announcement Time

In [13]:
fomc_2025_utc = {
    "January 2025": pd.Timestamp("2025-01-29 19:00:00", tz="UTC"),   # Winter (Standard Time)
    "March 2025": pd.Timestamp("2025-03-19 18:00:00", tz="UTC"),     # Summer (Daylight Saving)
    "May 2025": pd.Timestamp("2025-05-07 18:00:00", tz="UTC"),       # Summer (Daylight Saving)
    "June 2025": pd.Timestamp("2025-06-18 18:00:00", tz="UTC"),      # Summer (Daylight Saving)
    "July 2025": pd.Timestamp("2025-07-30 18:00:00", tz="UTC"),      # Summer (Daylight Saving)
    "September 2025": pd.Timestamp("2025-09-17 18:00:00", tz="UTC"), # Summer (Daylight Saving)
    "October 2025": pd.Timestamp("2025-10-29 18:00:00", tz="UTC"),   # Summer (Daylight Saving)
    "December 2025": pd.Timestamp("2025-12-10 19:00:00", tz="UTC"),  # Winter (Standard Time)
}

In [14]:
def filter_up_to_announcement(df, announcement_dict):

    if df.empty:
        return df
    last_timestamp = df["time"].iloc[-1]
    last_month_year = last_timestamp.strftime("%B %Y")

    if last_month_year in announcement_dict:
        cutoff_time = announcement_dict[last_month_year]
        filtered_df = df[df["time"] <= cutoff_time]
        return filtered_df
    else:
        print(
            f"Warning: '{last_month_year}' not found in the announcement dictionary"
        )
        return df

In [15]:
imp_df_fil = filter_up_to_announcement(imp_df,fomc_2025_utc)

In [16]:
imp_df_fil

,time,effr_avg_apr,effr_avg_may,effr_end_apr,effr_start_may,volume,effr_end_may,delta_effr,num_hikes,hikes_floor,hikes_decimal,prob_cut_100,prob_cut_75,prob_cut_50,prob_cut_25,prob_no_change,prob_hike_25,prob_hike_50,prob_hike_75,prob_hike_100
0,2025-01-01 23:00:00+00:00,4.185,4.1400,4.185,4.185,6,4.126875,-0.058125,-0.232500,-1.0,0.767500,0.0,0.0,0.0,0.232500,0.767500,0.0,0.0,0.0,0.0
1,2025-01-01 23:20:00+00:00,4.185,4.1450,4.185,4.185,17,4.133333,-0.051667,-0.206667,-1.0,0.793333,0.0,0.0,0.0,0.206667,0.793333,0.0,0.0,0.0,0.0
2,2025-01-01 23:32:00+00:00,4.185,4.1450,4.185,4.185,65,4.133333,-0.051667,-0.206667,-1.0,0.793333,0.0,0.0,0.0,0.206667,0.793333,0.0,0.0,0.0,0.0
3,2025-01-01 23:58:00+00:00,4.185,4.1450,4.185,4.185,2,4.133333,-0.051667,-0.206667,-1.0,0.793333,0.0,0.0,0.0,0.206667,0.793333,0.0,0.0,0.0,0.0
4,2025-01-02 00:00:00+00:00,4.190,4.1450,4.190,4.190,21,4.131875,-0.058125,-0.232500,-1.0,0.767500,0.0,0.0,0.0,0.232500,0.767500,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22985,2025-05-07 17:55:00+00:00,4.330,4.3250,4.330,4.330,1520,4.323542,-0.006458,-0.025833,-1.0,0.974167,0.0,0.0,0.0,0.025833,0.974167,0.0,0.0,0.0,0.0
22986,2025-05-07 17:57:00+00:00,4.330,4.3250,4.330,4.330,734,4.323542,-0.006458,-0.025833,-1.0,0.974167,0.0,0.0,0.0,0.025833,0.974167,0.0,0.0,0.0,0.0
22987,2025-05-07 17:58:00+00:00,4.330,4.3275,4.330,4.330,889,4.326771,-0.003229,-0.012917,-1.0,0.987083,0.0,0.0,0.0,0.012917,0.987083,0.0,0.0,0.0,0.0
22988,2025-05-07 17:59:00+00:00,4.330,4.3275,4.330,4.330,60,4.326771,-0.003229,-0.012917,-1.0,0.987083,0.0,0.0,0.0,0.012917,0.987083,0.0,0.0,0.0,0.0


## Export the Data as a CSV

In [17]:
import os
DATA_DIR = "../../data/processed/CME_implied_probabilities"
filename = "CME_IMP_MAY_2025.csv"
path = os.path.join(DATA_DIR, filename)
imp_df_fil.to_csv(path,index=False)